In [1]:
import pandas as pd
import numpy as np

# 1. LOAD & INSPECT

# Load the dataset into a table (df)
df = pd.read_csv("glass.csv")

# Quick check to see what we are working with
print("Data Shape:", df.shape)
print("Columns:", df.columns)
print(df.head())


Data Shape: (214, 10)
Columns: Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')
        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1


In [2]:
#2. DATA PREPROCESSING
# So basically, y is our assumed target column.
# We are just checking: Is Type == 1?
# If yes, .astype(int) converts it to 1. If no (types 2,3,7 etc), it becomes 0.
df["y"] = (df["Type"] == 1).astype(int)

# Drop the old 'Type' column.
# We don't want the model to see the answer key in the input features.
# Plus, we already extracted the info we needed into 'y'.
df = df.drop(columns=["Type"])

In [3]:
# TRAIN-TEST SPLIT FUNCTION
def custom_train_test_split(X, y, test_size=0.2, random_state=None):
    """
    Manually split data into training and testing sets.
    We shuffle the data randomly, then cut it into two pieces:
    - Training set (80% by default) for learning
    - Testing set (20% by default) for final evaluation

    random_state ensures we get the same shuffle every time (reproducibility).
    """
    # Set the random seed for reproducibility
    if random_state is not None:
        np.random.seed(random_state)

    # Get total number of samples
    n_samples = len(X)

    # Create an array of indices [0, 1, 2, ..., n_samples-1]
    indices = np.arange(n_samples)

    # Shuffle these indices randomly (like shuffling a deck of cards)
    np.random.shuffle(indices)

    # Calculate how many samples go to test set
    test_samples = int(n_samples * test_size)

    # Split the shuffled indices into test and train
    test_indices = indices[:test_samples]      # First 20% for testing
    train_indices = indices[test_samples:]     # Remaining 80% for training

    # Use these indices to split both X and y
    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test

In [4]:
#3.DATA SPLITTING

# Separate Questions (X) from Answers (y)
# .values turns the table into a raw matrix of numbers for the math functions
X = df.drop(columns=["y"]).values
y = df["y"].values

# Split into Study Material (Train) and Final Exam (Test)
# random_state=42 is the seed that helps in reproducibility.
# If we both use 42, we get the exact same split (same output for same input).
X_train, X_test, y_train, y_test = custom_train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
#STANDARD SCALER FUNCTIONS
def fit_scaler(X_train):
    """
    Calculate the mean and standard deviation for each feature in the training data.
    We need to remember these values to apply the same transformation to test data later.

    Think of it like learning the "average" and "spread" of each feature during training,
    so we can normalize new data the same way.
    """
    # Calculate mean for each column (feature)
    mean = np.mean(X_train, axis=0)

    # Calculate standard deviation for each column
    # This measures how spread out the values are
    std = np.std(X_train, axis=0)

    return mean, std

In [6]:
def transform_scaler(X, mean, std):
    return (X - mean) / std

   # Apply standardization: subtract mean and divide by standard deviation.
   # This centers data around 0 and scales it to have unit variance.

  #Formula: (X - mean) / std

   # Why? This prevents features with large values from dominating the learning process.
   #For example, a feature ranging 0-1000 shouldn't overpower one ranging 0-1.



In [7]:
#4.SCALING THE INPUTS

# This tries to bring all values to a comparable range (centered around 0).
# If we don't do this, larger values (like 1000) would dominate smaller ones (0.01),
# giving the wrong result. It also stops the sigmoid from getting stuck.

# First, learn the mean and std from training data
mean, std = fit_scaler(X_train)

# Apply the transformation to training data
X_train = transform_scaler(X_train, mean, std)

# Apply that same mean/std to test data
# IMPORTANT: We use training statistics on test data to avoid data leakage
X_test = transform_scaler(X_test, mean, std)

In [8]:
#5.SIGMOID FUNCTION
def sigmoid(z):
    # Squeezes any number (z) into a probability range between 0 and 1
    return 1 / (1 + np.exp(-z))

def predict_proba(X, w, b):
    # The '@' does matrix multiplication.
    # It calculates the weighted score for every row at once (features * weights).
    z = X @ w + b

    # Convert that raw score z into a probability p
    p = sigmoid(z)
    return p

In [9]:
#6. LOSS AND UPDATE

def loss(y, p):
    # Binary Cross Entropy (The Scorecard)
    # If y=1 (Type 1), it checks log(p). If y=0 (Not Type 1), it checks log(1-p).
    # The negative sign flips the result because log(0.9) is negative, but we want positive penalty points.
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def update_weights(X, y, w, b, lr):
    # 1. Get current predictions
    p = predict_proba(X, w, b)

    # 2. Check accuracy (Prediction - Actual)
    # This gives us the direction and magnitude of the mistake.
    error = p - y

    # 3. Update weights
    # If high penalty/error, weights are updated by a huge margin.
    # If low penalty, weights are updated only by a bit.
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)

    return w, b

In [10]:
#7. TRAINING LOOP

# Initialize weights as zeros (no opinion yet)
w = np.zeros(X_train.shape[1])
b = 0.0

lr = 0.1      # Learning rate (step size)
epochs = 100  # How many times we loop through the data

print(f"Starting Training... (Loss should go down)")
for i in range(epochs):
    # Run one step of learning
    w, b = update_weights(X_train, y_train, w, b, lr)

    # (Optional) Print loss every 10 steps to see if it's working
    if i % 10 == 0:
        p_train = predict_proba(X_train, w, b)
        current_loss = loss(y_train, p_train)
        print(f"Epoch {i}: Loss = {current_loss:.4f}")
#8. FINAL DECISION MAKING

def predict_label(p, threshold=0.5):
    # The model outputs a probability (p).
    # We decide: if p > threshold, we call it a '1' (Yes), otherwise '0' (No).
    return (p >= threshold).astype(int)

Starting Training... (Loss should go down)
Epoch 0: Loss = 0.6821
Epoch 10: Loss = 0.6106
Epoch 20: Loss = 0.5748
Epoch 30: Loss = 0.5530
Epoch 40: Loss = 0.5380
Epoch 50: Loss = 0.5271
Epoch 60: Loss = 0.5186
Epoch 70: Loss = 0.5118
Epoch 80: Loss = 0.5062
Epoch 90: Loss = 0.5016


In [11]:
# Final Test
print("\n--- Final Results ---")
p_test = predict_proba(X_test, w, b)
final_predictions = predict_label(p_test, threshold=0.5)

# Compare first 10 predictions vs actuals
print("Predictions:", final_predictions[:10])
print("Actuals:    ", y_test[:10])


--- Final Results ---
Predictions: [0 0 1 0 0 0 1 0 0 0]
Actuals:     [1 0 1 0 0 0 1 0 0 0]


In [12]:
# 1. How this differs from Perceptron

# The Perceptron is like a harsh light switch—it strictly snaps to 0 or 1, giving you no room for nuance.
# Logistic Regression is "softer"; instead of a hard decision, it gives you a probability score (like 0.75).
# This means it doesn't just tell you "Yes" or "No," it tells you how confident it is in that answer.


In [13]:
# 2. Why the Sigmoid function matters

#  The Perceptron uses a step function that jumps instantly, which makes it impossible to calculate a slope (derivative) for
#  learning. The Sigmoid function provides a smooth curve that is differentiable. This allows us to use gradient descent to calculate
#   exactly how much to adjust the weights to reduce errors.

In [14]:
# 3. What problem remains unsolved

# This model is still linear, meaning it can only separate data classes using a straight line. It fails completely on complex,
#  non-linear data patterns (like the XOR problem). The solution is to stack multiple neurons into layers to build a Neural Network.